# 🧠 Hands-On: Token Optimization in LLM Applications

**Program:** Brilian Sistem Informasi Bootcamp  
**Session 26:** Token Optimization

## 🎯 Objective
In this hands-on session, you will learn how to optimize token usage when working with Large Language Models (LLMs).

## 📌 Why Token Optimization Matters
- 💰 Cost: More tokens = higher cost
- ⚡ Speed: Larger prompts = slower response
- 🧠 Context: Too much input can reduce model focus

## 🛠️ What You Will Learn
1. Estimate token usage before sending prompts
2. Split long text using chunking
3. Use retrieval to send only relevant content

## 📊 Rule of Thumb
- 1 token ≈ 4 characters (English text)
- Longer prompts → more tokens → more cost

## ⚙️ Setup

> ⚠️ **IMPORTANT — Never hardcode your API keys.** Always use Google Colab Secrets.

**How to set it up:**
1. Click the 🔑 icon on the left sidebar in Colab
2. Add the following secret (Toggle **Notebook access** ON):
   - `GOOGLE_API_KEY` — your Gemini API key from Google AI Studio

In [ ]:
import math
from dataclasses import dataclass

def estimate_tokens(text: str):
    approx_tokens = max(1, math.ceil(len(text) / 4))
    approx_words = len(text.split())
    return {
        "characters": len(text),
        "words": approx_words,
        "estimated_tokens": approx_tokens,
        "rule_of_thumb": "Approximation based on 1 token ~= 4 characters"
    }

def chunk_text(text: str, target_tokens: int = 500, overlap_tokens: int = 50):
    approx_words = max(1, target_tokens)
    overlap_words = max(0, overlap_tokens)
    words = text.split()

    chunks = []
    start = 0
    while start < len(words):
        end = min(len(words), start + approx_words)
        chunk = " ".join(words[start:end]).strip()
        if chunk:
            chunks.append(chunk)
        if end >= len(words):
            break
        start = max(start + 1, end - overlap_words)
    return chunks

@dataclass
class KnowledgeDocument:
    doc_id: str
    title: str
    content: str

KNOWLEDGE_BASE = [
    KnowledgeDocument(
        "doc-01",
        "Azure OpenAI Authentication",
        "Azure OpenAI requires endpoint, deployment name, and API key."
    ),
    KnowledgeDocument(
        "doc-02",
        "Embeddings",
        "Embeddings convert text into vectors for semantic search."
    ),
    KnowledgeDocument(
        "doc-03",
        "Large Context Strategy",
        "Use chunking, retrieval, and summarization for large prompts."
    ),
]

## 🔍 Part 1: Token Estimation

Before sending text to an LLM, we should estimate how many tokens it will use.

### ❓ Why?
- Prevent exceeding token limits
- Control cost
- Design efficient prompts

### 🧪 Task
Compare token usage between:
- Short prompt
- Long prompt

In [ ]:
short_text = "Explain Azure OpenAI briefly."

long_text = """
Azure OpenAI is a cloud-based AI service that allows developers to use advanced language models
for various tasks including summarization, text generation, and embeddings. It integrates with
Azure services for scalability and security.
"""

print("Short prompt:")
print(estimate_tokens(short_text))

print("\nLong prompt:")
print(estimate_tokens(long_text))

Short prompt:
{'characters': 29, 'words': 4, 'estimated_tokens': 8, 'rule_of_thumb': 'Approximation based on 1 token ~= 4 characters'}

Long prompt:
{'characters': 237, 'words': 33, 'estimated_tokens': 60, 'rule_of_thumb': 'Approximation based on 1 token ~= 4 characters'}


### 🤔 Reflection
- Which text uses more tokens?
- How does prompt length affect cost?
- Can we make prompts shorter but still clear?

## ✂️ Part 2: Chunking

When text is too long, we split it into smaller pieces called **chunks**.

### ❓ Why Chunking?
- Avoid sending very large prompts
- Improve processing efficiency
- Enable retrieval-based workflows

### ⚙️ Key Parameters
- `target_tokens`: size of each chunk
- `overlap_tokens`: repeated context between chunks

---

### 🧪 Task
Try different chunk sizes and observe the results

In [ ]:
sample_text = """
Large language models process text, but large prompts increase cost and reduce efficiency.
Chunking splits text into smaller parts, while retrieval selects only relevant chunks.
""" * 10

chunks_100 = chunk_text(sample_text, target_tokens=100, overlap_tokens=10)
chunks_50 = chunk_text(sample_text, target_tokens=50, overlap_tokens=10)

print("Chunks (100 tokens):", len(chunks_100))
print("First chunk:\n", chunks_100[0])

print("\nChunks (50 tokens):", len(chunks_50))
print("First chunk:\n", chunks_50[0])

Chunks (100 tokens): 3
First chunk:
 Large language models process text, but large prompts increase cost and reduce efficiency. Chunking splits text into smaller parts, while retrieval selects only relevant chunks. Large language models process text, but large prompts increase cost and reduce efficiency. Chunking splits text into smaller parts, while retrieval selects only relevant chunks. Large language models process text, but large prompts increase cost and reduce efficiency. Chunking splits text into smaller parts, while retrieval selects only relevant chunks. Large language models process text, but large prompts increase cost and reduce efficiency. Chunking splits text into smaller parts, while retrieval selects only relevant chunks.

Chunks (50 tokens): 6
First chunk:
 Large language models process text, but large prompts increase cost and reduce efficiency. Chunking splits text into smaller parts, while retrieval selects only relevant chunks. Large language models process text, 

### 🤔 Reflection
- Which configuration produces more chunks?
- What happens if chunks are too small?
- What happens if chunks are too large?

👉 Insight:
Chunking is a trade-off between **efficiency** and **context completeness**

## 🔎 Part 3: Retrieval (Reduce Tokens!)

Instead of sending all documents to the model, we send only the most relevant ones.

### ❓ Why Retrieval?
- Reduces input tokens
- Improves model focus
- Scales better for large datasets

---

### 🧠 Idea
❌ Without retrieval:
Send ALL documents → expensive

✅ With retrieval:
Send only relevant documents → efficient

In [ ]:
def simple_retrieve(query, docs, top_k=2):
    query_words = set(query.lower().split())
    scored = []

    for doc in docs:
        doc_words = set(doc.content.lower().split())
        score = len(query_words.intersection(doc_words))
        scored.append((score, doc))

    scored.sort(reverse=True, key=lambda x: x[0])
    return [doc for score, doc in scored[:top_k]]

query = "How to handle large prompts?"

results = simple_retrieve(query, KNOWLEDGE_BASE, top_k=1)

for doc in results:
    print(doc.title)
    print(doc.content)

Large Context Strategy
Use chunking, retrieval, and summarization for large prompts.


## 🧪 Final Exercise

### 🎯 Goal
Compare token usage between:
1. Sending ALL documents
2. Sending only retrieved documents

---

### 📝 Instructions
- Estimate tokens for full content
- Estimate tokens for retrieved content
- Compare results

👉 Which is more efficient?

In [ ]:
user_question = "How to reduce token usage?"

# All content
all_text = " ".join([doc.content for doc in KNOWLEDGE_BASE])
print("All content tokens:")
print(estimate_tokens(all_text))

# Retrieved content
retrieved_docs = simple_retrieve(user_question, KNOWLEDGE_BASE, top_k=1)
retrieved_text = " ".join([doc.content for doc in retrieved_docs])

print("\nRetrieved content tokens:")
print(estimate_tokens(retrieved_text))

All content tokens:
{'characters': 181, 'words': 25, 'estimated_tokens': 46, 'rule_of_thumb': 'Approximation based on 1 token ~= 4 characters'}

Retrieved content tokens:
{'characters': 61, 'words': 9, 'estimated_tokens': 16, 'rule_of_thumb': 'Approximation based on 1 token ~= 4 characters'}


## 🎯 Key Takeaways

✅ Estimate tokens before sending prompts  
✅ Use chunking for large text  
✅ Use retrieval to send only relevant data  

---

## 💡 Final Insight
Token optimization is not just about saving cost —  
it improves performance, speed, and scalability of LLM applications.

## 🚀 Part 4: Integration with Google Gemini

Now that you understand token optimization, let's see how to actually use these optimized prompts with the **Google Gemini API**.

### 🛠️ Setup Client
We will use the `google.generativeai` Python library to interact with Google Gemini.

In [ ]:
# The google-generativeai library is usually pre-installed in Colab.
# If not, you can uncomment the following line to install it:
#!pip install google-generativeai -q

In [ ]:
import google.generativeai as genai
from google.colab import userdata
from IPython.display import Markdown, display

# Retrieve secrets from Colab
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

# Initialize the Gemini model
gemini_model = genai.GenerativeModel('gemini-3.1-flash-lite') # Using a common Gemini Flash model

# Example of sending the 'retrieved' optimized text
# For Gemini, we typically send messages as a list of dictionaries with 'role' and 'parts'
response = gemini_model.generate_content(
    [{"role": "user", "parts": [f"Context: {retrieved_text}\n\nQuestion: {user_question}"]}]
)

print("Response from Google Gemini:")
display(Markdown(response.text))

Response from Google Gemini:


To reduce token usage in Azure OpenAI, you need to focus on optimizing the "input" (prompt) and the "output" (completion). Here are the most effective strategies:

### 1. Optimize Your Prompts
*   **Be Concise:** Remove unnecessary instructions or conversational "fluff." If you provide examples, ensure they are high-quality rather than high-quantity.
*   **System Prompt Tuning:** Move repetitive instructions into the System Prompt. Because system prompts are processed every turn, keep them as short as possible while remaining clear.
*   **Shorten Context:** Only include the most relevant parts of the conversation history. If the chat history grows too long, summarize it instead of sending the entire transcript.

### 2. Control the Output Size
*   **Set `max_tokens` carefully:** By default, models may try to generate long responses. If you know the task only requires a short answer (e.g., classification or a one-word label), cap the `max_tokens` parameter strictly.
*   **Request Specific Formats:** If you are asking for data, request JSON or CSV formats rather than prose. Prose contains many "filler" words that consume tokens without providing functional value.
*   **Stop Sequences:** Define `stop` sequences (e.g., `\n` or `###`) to force the model to stop generating as soon as it reaches the end of the intended answer, preventing "run-on" sentences.

### 3. Leverage Model Capabilities
*   **Use the Right Model for the Job:** Not every task requires GPT-4o. If you are doing simple tasks like sentiment analysis, keyword extraction, or basic summarization, use **GPT-4o-mini**. It is significantly cheaper and more efficient for simple tasks.
*   **Avoid "Over-Prompting":** Don't use a massive model if a smaller one can handle the logic. 

### 4. Architectural Strategies
*   **Caching (Semantic Cache):** If your application receives repetitive queries (e.g., "What are your business hours?"), store the request/response pair in a database (like Redis). Check your cache before sending a request to the API.
*   **Summarization instead of History:** Instead of sending the last 20 messages, use a background process to summarize the conversation into 2-3 sentences and send the summary as the context for the next turn.

### 5. Review "Chat Completion" Overhead
*   Every time you send a list of messages (the `messages` array), the entire history is sent as input tokens. 
    *   **Bad practice:** Sending the full history including system messages for every single turn.
    *   **Good practice:** Send only the system message, the last 2-3 exchanges, and a concise summary of the earlier conversation.

### Summary Checklist for Implementation:
1.  **Check `max_tokens`**: Is it set to the minimum reasonable value?
2.  **Audit History**: Are you sending unnecessary conversation history?
3.  **Evaluate Model**: Can this be done by `GPT-4o-mini` instead of `GPT-4o`?
4.  **Implement Caching**: Can you serve the result from a local database instead of calling the API?